In [13]:
import torch
import numpy as np
import pandas as pd

from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from sklearn.decomposition import PCA

In [14]:
from pymongo import MongoClient
from dotenv import load_dotenv
import os

# Load dataset

In [15]:
load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")

client = MongoClient(MONGO_URI)
db=client["PolymerPrediction"]
collection=db["biceranoTg"]
target="Tg"
#print(collection.count_documents({}))

In [16]:
docs = list(collection.find({}))
df = pd.DataFrame(docs)

# espando il dizionario "properties"
prop_df = pd.json_normalize(df["properties"])

# integro nel dataframe
df = pd.concat([df.drop(columns=["properties"]), prop_df], axis=1)

# rimuovo righe senza SMILES, Tg o densità o con SMILES malformato
df = df.dropna(subset=["smiles", target])
df = df[~df["smiles"].str.contains(r"\[\[", na=False)].copy()
df.describe()

ids = df["_id"].astype(str).rename("_id")

# Load ChemBERTa

In [17]:
model_name = "seyonec/ChemBERTa-zinc-base-v1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

model.eval()

Loading weights: 100%|████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 12021.07it/s]
RobertaModel LOAD REPORT from: seyonec/ChemBERTa-zinc-base-v1
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(767, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-5): 6 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dropout): 

# Embedding extraction from SMILES

In [18]:
def chemberta_embedding(smiles_list):

    embeddings = []

    for smiles in tqdm(smiles_list):

        inputs = tokenizer(
            smiles,
            return_tensors="pt",
            truncation=True,
            padding=True
        )

        with torch.no_grad():
            outputs = model(**inputs)

        emb = outputs.last_hidden_state.mean(dim=1)

        embeddings.append(emb.squeeze().numpy())

    return np.array(embeddings)

# Full size embeddings 

In [19]:
smiles_list = df["smiles"].tolist()
emb_array=chemberta_embedding(smiles_list)
X_embedding = pd.DataFrame(emb_array, columns=[f"emb_{i}" for i in range(emb_array.shape[1])])


print(X_embedding.shape)

100%|████████████████████████████████████████████████████████████████████████████████| 304/304 [00:10<00:00, 29.49it/s]

(304, 768)


In [20]:
df_out = pd.concat([
    ids.reset_index(drop=True),
    X_embedding.reset_index(drop=True)
], axis=1)

df_out.to_csv("chemberta_embeddings.csv", index=False)

# Reduced embeddings (PCA, 50 columns)

In [21]:
## Dimensionality reduction
pca = PCA(n_components=50)

X_emb_reduced = pca.fit_transform(X_embedding)
X_emb_reduced = pd.DataFrame(X_emb_reduced, columns=[f"pc_{i}" for i in range(X_emb_reduced.shape[1])])

print(X_emb_reduced.shape)

(304, 50)


In [22]:
df_out_red = pd.concat([
    ids.reset_index(drop=True),
    X_emb_reduced.reset_index(drop=True)
], axis=1)

df_out_red.to_csv("chemberta_embeddings_reduced.csv", index=False)

# Sanitized embeddings (removed wildcards)

In [24]:
clean_smiles_list = [s.replace("[*]", "") for s in smiles_list]
clean_emb_array=chemberta_embedding(clean_smiles_list)
clean_emb=pd.DataFrame(clean_emb_array, columns=[f"emb_{i}" for i in range(clean_emb_array.shape[1])])

df_out_clean=pd.concat([
    ids.reset_index(drop=True),
    clean_emb.reset_index(drop=True)
], axis=1)

df_out_clean.to_csv("chemberta_clean_embeddings.csv", index=False)

## Dimensionality reduction
pca = PCA(n_components=50)

X_emb_clean_reduced = pca.fit_transform(clean_emb)
X_emb_clean_reduced=pd.DataFrame(X_emb_clean_reduced, columns=[f"pc_{i}" for i in range(X_emb_clean_reduced.shape[1])])

print(X_emb_clean_reduced.shape)

df_out_clean_red=pd.concat([
    ids.reset_index(drop=True),
    X_emb_clean_reduced.reset_index(drop=True)
], axis=1)

df_out_clean_red.to_csv("chemberta_clean_embeddings_reduced.csv", index=False)

100%|████████████████████████████████████████████████████████████████████████████████| 304/304 [00:09<00:00, 33.27it/s]


(304, 50)


# Less aggressive PCA (200 columns)

## Unsanitized SMILES

In [27]:

## Dimensionality reduction
pca = PCA(n_components=200)

X_emb_less_reduced = pca.fit_transform(X_embedding)
X_emb_less_reduced = pd.DataFrame(X_emb_less_reduced, columns=[f"pc_{i}" for i in range(X_emb_less_reduced.shape[1])])

print(X_emb_less_reduced.shape)

df_out_less_red = pd.concat([
    ids.reset_index(drop=True),
    X_emb_less_reduced.reset_index(drop=True)
], axis=1)

df_out_less_red.to_csv("chemberta_embeddings_less_reduced.csv", index=False)

(304, 200)


## Sanitized SMILES

In [28]:


X_emb_clean_less_reduced = pca.fit_transform(clean_emb)
X_emb_clean_less_reduced=pd.DataFrame(X_emb_clean_less_reduced, columns=[f"pc_{i}" for i in range(X_emb_clean_less_reduced.shape[1])])

print(X_emb_clean_less_reduced.shape)

df_out_clean_less_red=pd.concat([
    ids.reset_index(drop=True),
    X_emb_clean_less_reduced.reset_index(drop=True)
], axis=1)

df_out_clean_less_red.to_csv("chemberta_clean_embeddings_less_reduced.csv", index=False)

(304, 200)
